# 7-to-1 |Y⟩ State Distillation (Steane Code)

Reference: `processing/Steane_distillation.png` Figure 46(c)

**Circuit structure:**
- W1, W2, W3: injected |Y⟩ (noisy)
- W4: fault-tolerant |+⟩
- A: ancilla |Y⟩ (reused per π/4 rotation)
- 4 sequential Z product measurements
- End: MX on W1-W3 (post-select), S†+MX on W4 (output)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.qec_code.surface_code.unrotated import (
    UnrotatedSurfaceCode,
    UnrotatedSurfaceCodeExtractionBlock,
    UnrotatedMultiPatchCoupler,
    UnrotatedSurfaceCodeLogicalOpSet,
)
from src.ir.qec_system import QECSystem
from src.ir.builder import CircuitBuilder
from src.ir.tracker import SyndromeTracker
import numpy as np

## Step 1: Single π/4 Rotation Test

Layout from `processing/Distillation layout.jpeg`:
```
W1(-2,0)    W3(10,0)
W2(-2,8)    W4(10,8)   ← IDLE
W5(-2,16)              ← Magic state
```

Protocol:
1. W1,W2,W3,W4 init Z (working qubits)
2. W5 init |+⟩ then transversal S → |Y⟩ (magic state)
3. ZZZZ measurement on {W1, W2, W3, W5} (W4 idle)
4. MX on W5, MZ on W1-W4, MX on coupler data
5. Expected: 4 logical observables (Z on W1-W4 preserved)

In [2]:
d = 3
rounds = d
center = 6.0

# --- Patch Setup ---
patch_layout = {
    'W1': (-2, 0),    # left-top
    'W3': (10, 0),    # right-top
    'W2': (-2, 8),    # left-mid
    'W4': (10, 8),    # right-mid (idle for first meas, active for others)
    'W5': (-2, 16),   # left-bottom (magic state, reused)
}

system = QECSystem()
for name, offset in patch_layout.items():
    p = UnrotatedSurfaceCode(distance=d)
    p.transpose_coords()
    system.add_patch(p, name=name, offset=offset)

op_set = UnrotatedSurfaceCodeLogicalOpSet(UnrotatedSurfaceCodeExtractionBlock)

print("Patch bounds:")
for name in patch_layout:
    print(f"  {name}: {system.patches[name][0]._get_bounds()}")
print(f"\nSystem: {system.num_qubits} qubits, {system.num_logicals} logicals")

# --- Tracker + Builder ---
tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=True)
builder.write_coordinates()

# --- Init working qubits in Z ---
working_data = {q: 'Z' for q in system.data_indices
                if system.index_to_owner_map[q] in ('W1','W2','W3','W4')}
builder.initialize(init_dict=working_data, n=system.num_qubits)

# --- Helper: get all global qubit indices for a patch ---
def get_patch_qubit_indices(patch_name):
    indices = set()
    for local_idx in system.local_to_global_map[patch_name].values():
        indices.add(local_idx)
    return indices

# --- Helper: inject |Y⟩ into W5 ---
def init_magic_state():
    w5_data = {q: 'X' for q in system.data_indices if system.index_to_owner_map[q] == 'W5'}
    builder.initialize(init_dict=w5_data, n=system.num_qubits)
    op_set.fold_transversal_s(builder, system.patches['W5'][0])

# --- Helper: one Z product measurement cycle ---
def do_zzzz_measurement(subset, coupler_name):
    system.register_coupler(UnrotatedMultiPatchCoupler(),
        patch_names=subset, name=coupler_name,
        path_axis='vertical', center_axis=center)
    n = system.num_qubits
    if n > tracker.num_qubits:
        tracker.expand(n - tracker.num_qubits)
    builder.write_coordinates()
    
    builder.activate_coupler(coupler_name)
    cp = system.coupler_patches[coupler_name]
    cd = [system.local_to_global_map[coupler_name][q] for q in cp.data_indices]
    builder.initialize(init_dict={q: 'X' for q in cd}, n=n)
    
    se = UnrotatedSurfaceCodeExtractionBlock(system)
    builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)
    
    builder.deactivate_coupler(coupler_name)
    system.remove_coupler(coupler_name)

# --- Init W5 as |Y⟩ + stabilize ---
init_magic_state()
se = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

# --- 4 sequential ZZZZ measurements ---
subsets = [
    ['W1', 'W2', 'W3', 'W5'],
    ['W1', 'W2', 'W4', 'W5'],
    ['W1', 'W3', 'W4', 'W5'],
    ['W2', 'W3', 'W4', 'W5'],
]

for i, subset in enumerate(subsets):
    do_zzzz_measurement(subset, f'meas_{i+1}')
    print(f"  Meas {i+1}: ZZZZ on {subset} — done")
    
    # Re-inject W5 between measurements
    if i < len(subsets) - 1:
        # Clear old measurement records for W5 qubits (prevents non-deterministic detectors)
        tracker.reset_records_for_qubits(get_patch_qubit_indices('W5'))
        init_magic_state()
        tracker.expected_num_logicals += 1  # W5's logical DOF restored
        se = UnrotatedSurfaceCodeExtractionBlock(system)
        builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

# --- Final readout ---
measure_dict = {}
for q in system.data_indices:
    owner = system.index_to_owner_map.get(q)
    if owner in ('W1', 'W2', 'W3', 'W4'):
        measure_dict[q] = 'Z'
    elif owner == 'W5':
        measure_dict[q] = 'X'
builder.apply_data_readout(final_measurements=measure_dict)

circuit = builder.circuit
print(f"\nCircuit: {circuit.num_qubits} qubits, {circuit.num_detectors} det, {circuit.num_observables} obs")

try:
    dem = circuit.detector_error_model(decompose_errors=True)
    print(f"DEM OK: {dem.num_detectors} det, {dem.num_observables} obs")
except Exception as e:
    print(f"DEM error: {type(e).__name__}: {str(e)[:200]}")

# circuit.diagram("detslice-with-ops-svg")

Patch bounds:
  W1: (-2.0, 2, 0.0, 4)
  W3: (10.0, 14, 0.0, 4)
  W2: (-2.0, 2, 8.0, 12)
  W4: (10.0, 14, 8.0, 12)
  W5: (-2.0, 2, 16.0, 20)

System: 125 qubits, 5 logicals
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
  Meas 1: ZZZZ on ['W1', 'W2', 'W3', 'W5'] — done
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
  Meas 2: ZZZZ on ['W1', 'W2', 'W4', 'W5'] — done
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
  Meas 3: ZZZZ on ['W1', 'W3', 'W4', 'W5'] — done
Applying first round of syndrome extraction...
Applying rest rounds of syndrome extraction...
Applying first round

In [4]:
circuit

stim.Circuit('''
    QUBIT_COORDS(-2, 1) 0
    QUBIT_COORDS(-2, 3) 1
    QUBIT_COORDS(-2, 0) 2
    QUBIT_COORDS(-2, 2) 3
    QUBIT_COORDS(-2, 4) 4
    QUBIT_COORDS(-1, 0) 5
    QUBIT_COORDS(-1, 2) 6
    QUBIT_COORDS(-1, 4) 7
    QUBIT_COORDS(-1, 1) 8
    QUBIT_COORDS(-1, 3) 9
    QUBIT_COORDS(0, 1) 10
    QUBIT_COORDS(0, 3) 11
    QUBIT_COORDS(0, 0) 12
    QUBIT_COORDS(0, 2) 13
    QUBIT_COORDS(0, 4) 14
    QUBIT_COORDS(1, 0) 15
    QUBIT_COORDS(1, 2) 16
    QUBIT_COORDS(1, 4) 17
    QUBIT_COORDS(1, 1) 18
    QUBIT_COORDS(1, 3) 19
    QUBIT_COORDS(2, 1) 20
    QUBIT_COORDS(2, 3) 21
    QUBIT_COORDS(2, 0) 22
    QUBIT_COORDS(2, 2) 23
    QUBIT_COORDS(2, 4) 24
    QUBIT_COORDS(10, 1) 25
    QUBIT_COORDS(10, 3) 26
    QUBIT_COORDS(10, 0) 27
    QUBIT_COORDS(10, 2) 28
    QUBIT_COORDS(10, 4) 29
    QUBIT_COORDS(11, 0) 30
    QUBIT_COORDS(11, 2) 31
    QUBIT_COORDS(11, 4) 32
    QUBIT_COORDS(11, 1) 33
    QUBIT_COORDS(11, 3) 34
    QUBIT_COORDS(12, 1) 35
    QUBIT_COORDS(12, 3) 36
    QUBIT